# 第十课｜spike 为什么需要排队？

上一课一个 engine 要轮流服务很多 state，所以它不一定能在 spike 到来的同一瞬间完成所有后续工作。今天只解决：
> **当事件到达速度暂时快于消费速度时，怎样既保持顺序，又不悄悄丢事件？**

主要新概念：**有界先进先出队列（bounded First-In, First-Out queue, FIFO）**。


## 1. 概念账本

**已经知道：** spike 是离散事件；一个共享 engine 可能忙于处理其他 neuron。

**今天学习：** event queue、FIFO、full/empty，以及 backpressure。

**只预告：** 下一课才讨论一个 spike 怎样找到它的下游 synapse。


## 2. event 是什么？

这里把一个 spike 简化成一个很小的 **event（事件记录）**。最小版本只带 `source_id`：谁 spike 了。后续可以加时间、类型或其他 metadata，但本课不需要。


## 3. FIFO 保证什么？

**先进先出（First-In, First-Out, FIFO）**：最早进入队列的 event 最早离开。

如果依次进入 `2, 5, 7`，正常弹出顺序也必须是 `2, 5, 7`。FIFO 解决的是**暂存与顺序**，不是决定这些 spike 的生物学意义。


## 4. 有界队列为什么还需要 backpressure？

真实硬件 queue 容量有限。满了以后，producer 不能假装下一条 event 已经被接收。

**背压（backpressure）**：consumer/queue 用一个控制条件告诉 producer“现在不能再接收，请先保持/等待”。在本课的 Python 模型里，用 `accepted=False` 表示这件事。

```mermaid
flowchart LR
 P["spike producer"] -->|push event| Q["bounded FIFO"]
 Q -->|pop event| C["consumer"]
 Q -. "full / not ready" .-> P
```


## 5. Run：容量只有 3 的 queue

先预测第四个 event `9` 会怎样，再运行。


In [ ]:
from collections import deque

capacity = 3
q = deque()

def show(action):
    print(f'{action:18s} queue={list(q)} full={len(q) == capacity} empty={len(q) == 0}')

for event in [2, 5, 7, 9]:
    if len(q) < capacity:
        q.append(event)
        show(f'accepted spike {event}')
    else:
        show(f'blocked spike {event}')

while q:
    event = q.popleft()
    show(f'consumed spike {event}')


## 6. Observe

前三个 event 被接收；queue full 后，第四个 event 没有被偷偷覆盖前面的数据。之后 pop 顺序仍保持最初的到达顺序。

**重要：** “blocked” 不等于“允许丢弃”。正式接口需要 producer 在 ready 之前保持 event，或由更上游保存它。


## 7. Try It：consumer 变慢

把 capacity 改成 2。先预测哪个 push 最先被阻塞。然后尝试“push 两个 → pop 一个 → 再 push 一个”，观察 backpressure 如何解除。


## 8. 作业

完成 `exercises/lesson10_fifo.py` 的 `fifo_push(...)` 与 `fifo_pop(...)`。公开检查覆盖：

- FIFO ordering；
- empty pop；
- full 时拒绝新 event，且已有 queue 不被破坏。

```bash
uv run pytest exercises/checks/check_lesson10.py -q
```


## 9. AI Task

让 AI 为容量 2 的 FIFO 列出一条最短操作序列，同时覆盖 empty、non-empty、full、backpressure 四种状态。要求先列预期 queue，再给代码。


## 10. Human Check

不用 AI，你应该能解释：FIFO 为什么保序；full 与 empty 各代表什么；backpressure 为什么比“满了就覆盖旧 spike”更安全；为什么 queue 只是事件运输机制，不应该偷偷改变 neuron semantics。


## 11. Engineering Handoff

本课对齐 `MOD-005 spike_fifo` 与 `IF-SPIKE-QUEUE` 的概念契约，以及 T-007 FIFO ordering / T-008 FIFO backpressure；还没有实现正式 RTL FIFO。


## 12. 项目追踪 Project Trace

- Lesson: `LSN-010`
- Mapping: `RMD-007A / RMD-009` teaching precursor
- Module context: `MOD-005`
- Test context: `T-007 / T-008`


## 13. Exit Ticket

你能用自己的话说明 bounded FIFO 的 ordering 与 backpressure，并判断一个 full queue 为什么必须显式拒绝或延迟新 event。
